# Persistence in LangGraph

Persistence in LangGraph is the ability to save the execution state of a workflow at different stages using a **checkpointer**. Instead of losing all progress when a workflow finishes or is interrupted, LangGraph stores the current state, allowing it to be resumed, inspected, or continued later. Each conversation or execution is identified using a unique `thread_id`, which acts as a key to retrieve the corresponding saved state. Checkpointers such as `InMemorySaver` (for development) or database-backed storage (for production) automatically save the graph state after each node execution without requiring manual state management.

## Benefits of Persistence in LangGraph

### 1. Fault Tolerance
Persistence enables workflows to recover from unexpected failures such as application crashes, network interruptions, or system restarts. Since the graph state is automatically checkpointed after each node, execution can resume from the last saved checkpoint instead of restarting from the beginning, improving reliability and reducing computation time.

### 2. Time Travel
LangGraph allows developers to revisit previous checkpoints of a workflow, a feature known as **time travel**. This makes it possible to inspect historical states, replay executions, compare different stages of a workflow, and debug issues by observing how the state evolved over time.

### 3. State Updates
Saved graph states can be modified before resuming execution. Developers can update variables, correct errors, or inject new information into the workflow without rerunning all previous nodes. This flexibility is particularly useful during debugging, testing, and dynamic workflow management.

### 4. Human-in-the-Loop (HITL)
Persistence enables workflows to pause execution while waiting for human input, approval, or correction. Once the required input is provided, the workflow resumes from the exact point where it was paused rather than starting over. This capability is essential for applications requiring human oversight, such as document review, customer support, and approval systems.

### 5. Memory
Persistence provides long-term memory by storing workflow states across multiple interactions. Using unique `thread_id`s, LangGraph can maintain separate conversation histories or execution contexts for different users. This enables AI agents to remember previous interactions, preserve context across sessions, and deliver more personalized and coherent responses.
Persistence is one of LangGraph's most powerful features because it transforms workflows from stateless function calls into stateful, resumable applications capable of handling complex, long-running tasks.

In [1]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
llm=ChatOllama(model="llama3.1:8b", temperature=0.2,)

In [3]:
class joke_state(TypedDict):
    topic:str
    joke:str
    explanation:str


In [4]:
def generate_joke(state):
    print("Entered generate_joke")

    prompt = f"generate a joke on the topic {state['topic']}"
    response = llm.invoke(prompt).content

    print("Leaving generate_joke")
    return {"joke": response}

In [5]:
def explain_joke(state):
    print("Entered explain_joke")

    prompt = f"write an explanation for the joke - {state['joke']}"
    response = llm.invoke(prompt).content

    print("Leaving explain_joke")
    return {"explanation": response}

In [6]:
graph=StateGraph(joke_state)
graph.add_node("generate_joke", generate_joke)
graph.add_node("explain_joke", explain_joke)

graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "explain_joke")
graph.add_edge("explain_joke", END)
checkpointer=InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)

In [7]:
config1 = {"configurable": {"thread_id": "1"}}
result=workflow.invoke({'topic':'pizza'},config=config1)
print(result)

Entered generate_joke
Leaving generate_joke
Entered explain_joke
Leaving explain_joke
{'topic': 'pizza', 'joke': "Here's one:\n\nWhy was the pizza in a bad mood?\n\nBecause it was feeling a little crusty! (get it?)", 'explanation': 'Here\'s an explanation of the joke:\n\nThe joke is a play on words, using puns to create humor. The punchline "feeling a little crusty" has a double meaning. In one sense, being "crusty" can describe someone or something that\'s feeling irritable or in a bad mood. However, in the context of pizza, the word "crusty" also refers to the outer layer of bread on a pizza, which is literally called the "crust".\n\nThe joke relies on this wordplay to create a pun, where the phrase "feeling a little crusty" can be interpreted both literally (as in, feeling bad) and figuratively (as in, referring to the pizza\'s crust). The humor comes from the unexpected twist on the usual meaning of the word, creating a clever and silly connection between the setup and the punchlin

In [8]:
workflow.get_state(config1)


StateSnapshot(values={'topic': 'pizza', 'joke': "Here's one:\n\nWhy was the pizza in a bad mood?\n\nBecause it was feeling a little crusty! (get it?)", 'explanation': 'Here\'s an explanation of the joke:\n\nThe joke is a play on words, using puns to create humor. The punchline "feeling a little crusty" has a double meaning. In one sense, being "crusty" can describe someone or something that\'s feeling irritable or in a bad mood. However, in the context of pizza, the word "crusty" also refers to the outer layer of bread on a pizza, which is literally called the "crust".\n\nThe joke relies on this wordplay to create a pun, where the phrase "feeling a little crusty" can be interpreted both literally (as in, feeling bad) and figuratively (as in, referring to the pizza\'s crust). The humor comes from the unexpected twist on the usual meaning of the word, creating a clever and silly connection between the setup and the punchline.'}, next=(), config={'configurable': {'thread_id': '1', 'checkp

In [9]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': "Here's one:\n\nWhy was the pizza in a bad mood?\n\nBecause it was feeling a little crusty! (get it?)", 'explanation': 'Here\'s an explanation of the joke:\n\nThe joke is a play on words, using puns to create humor. The punchline "feeling a little crusty" has a double meaning. In one sense, being "crusty" can describe someone or something that\'s feeling irritable or in a bad mood. However, in the context of pizza, the word "crusty" also refers to the outer layer of bread on a pizza, which is literally called the "crust".\n\nThe joke relies on this wordplay to create a pun, where the phrase "feeling a little crusty" can be interpreted both literally (as in, feeling bad) and figuratively (as in, referring to the pizza\'s crust). The humor comes from the unexpected twist on the usual meaning of the word, creating a clever and silly connection between the setup and the punchline.'}, next=(), config={'configurable': {'thread_id': '1', 'check

In [10]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

Entered generate_joke
Leaving generate_joke
Entered explain_joke
Leaving explain_joke


{'topic': 'pasta',
 'joke': 'Here\'s one:\n\nWhy did the spaghetti go to therapy?\n\nBecause it was feeling a little "twisted"! (get it?)',
 'explanation': 'Here\'s an explanation of the joke:\n\nThe joke is a play on words, using a pun to create humor. The phrase "feeling a little twisted" has a double meaning here.\n\nIn one sense, "twisted" can refer to emotional distress or turmoil, implying that the spaghetti is feeling anxious or upset and therefore needs therapy. This is a common usage of the word in everyday language.\n\nHowever, "twisted" also has a literal meaning when applied to food like spaghetti: it can be twisted into various shapes, such as a knot or a curl. The joke relies on this second meaning to create a clever connection between the setup and punchline. By using the phrase "feeling a little twisted," the speaker is making a wordplay that links the idea of emotional distress with the physical characteristic of spaghetti being twisted.\n\nThe humor comes from the une

In [15]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1784de-7689-64fc-bfff-7c7d67b55994"}})

StateSnapshot(values={}, next=('__start__',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1784de-7689-64fc-bfff-7c7d67b55994'}}, metadata={'source': 'input', 'step': -1, 'parents': {}}, created_at='2026-07-05T08:45:41.540156+00:00', parent_config=None, tasks=(PregelTask(id='59a02f66-cae8-3df6-c3f6-e534021736ba', name='__start__', path=('__pregel_pull', '__start__'), error=None, interrupts=(), state=None, result={'topic': 'pizza'}),), interrupts=())

In [16]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f1784de-7689-64fc-bfff-7c7d67b55994"}})

Entered generate_joke
Leaving generate_joke
Entered explain_joke
Leaving explain_joke


{'topic': 'pizza',
 'joke': "Here's one:\n\nWhy was the pizza in a bad mood?\n\nBecause it was feeling a little crusty! (get it?)",
 'explanation': 'The joke is a play on words. The phrase "feeling a little crusty" has a double meaning here.\n\nIn everyday language, when someone says they\'re feeling "crusty", they usually mean that they\'re in a bad mood or irritable. But in the context of food, especially pizza, "crusty" refers to the outer layer of bread on the pizza, which is crispy and hard.\n\nThe joke relies on this wordplay to create a pun. It takes the common phrase "feeling a little crusty" and gives it a literal twist by connecting it to the pizza\'s physical characteristics (its crust). The punchline is humorous because it\'s a clever and unexpected connection between the setup ("Why was the pizza in a bad mood?") and the punchline ("Because it was feeling a little crusty!").'}

In [17]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': "Here's one:\n\nWhy was the pizza in a bad mood?\n\nBecause it was feeling a little crusty! (get it?)", 'explanation': 'The joke is a play on words. The phrase "feeling a little crusty" has a double meaning here.\n\nIn everyday language, when someone says they\'re feeling "crusty", they usually mean that they\'re in a bad mood or irritable. But in the context of food, especially pizza, "crusty" refers to the outer layer of bread on the pizza, which is crispy and hard.\n\nThe joke relies on this wordplay to create a pun. It takes the common phrase "feeling a little crusty" and gives it a literal twist by connecting it to the pizza\'s physical characteristics (its crust). The punchline is humorous because it\'s a clever and unexpected connection between the setup ("Why was the pizza in a bad mood?") and the punchline ("Because it was feeling a little crusty!").'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'ch

In [19]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1784fc-9a77-632f-8000-515d05a7c25e'}}

In [20]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1784fc-9a77-632f-8000-515d05a7c25e'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-07-05T08:59:10.613985+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, tasks=(PregelTask(id='1d099c6c-4fa0-8bed-d1f5-3552ed2c1472', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': "Here's one:\n\nWhy was the pizza in a bad mood?\n\nBecause it was feeling a little crusty! (get it?)", 'explanation': 'The joke is a play on words. The phrase "feeling a little crusty" has a double meaning here.\n\nIn everyday language, when someone says they\'re feeling "crusty", they usually mean that they\'re in a bad mood or